# Topic Labeling & Enrichment

Generate LLM-based labels and enriched descriptions for topics from **LDA, DTM, BERTopic, Top2Vec**.

Uses topic words from `results/{model}/temporal/{subject}/topic_word_evolution.csv`.

**Two Steps:**
1. **Overall Label & Enriched Description** — Combine all top words across all years → single label + rich description per topic
2. **Per-Year Simple Description** — For each year's top words → short description of what the topic looks like that year

In [1]:
import os
import re
import json
import time
import pickle
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [2]:
LIST_MODELS = ["lda", "dtm", "bertopic", "top2vec", "topicGpt"]
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../results")
CHECKPOINT_DIR = Path("../../models/labeling")

# LLM Configuration (LM Studio)
LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 4096

# Create checkpoint directories
for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        (CHECKPOINT_DIR / model / subject).mkdir(parents=True, exist_ok=True)

print(f"Models: {LIST_MODELS}")
print(f"Subjects: {LIST_SUBJECT}")
print(f"LLM: {LLM_MODEL} @ {LLM_API_URL}")

Models: ['lda', 'dtm', 'bertopic', 'top2vec', 'topicGpt']
Subjects: ['cs', 'math', 'physics']
LLM: mistralai/ministral-3-3b @ http://localhost:1234/v1/chat/completions


## LLM API Helper

In [3]:
def call_llm(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str:
    """Call LM Studio API with retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  LLM call failed after {max_retries} attempts: {e}")
                return ""

# Test connection
test_resp = call_llm("You are a helpful assistant.", "Say 'OK' if you can read this.")
print(f"LLM connection test: {test_resp[:100]}")

LLM connection test: OK! How can I assist you today?


## Checkpoint Utilities

In [4]:
def save_checkpoint(data, name: str, model: str, subject: str):
    """Save checkpoint to disk."""
    path = CHECKPOINT_DIR / model / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, model: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = CHECKPOINT_DIR / model / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## JSON Parsing Helper

In [5]:
def clean_and_parse_json(response: str) -> dict:
    """Parse JSON from LLM response, handling markdown wrappers."""
    text = re.sub(r"```json\s*|```", "", response).strip()
    
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return None
    
    json_str = text[start:end+1]
    json_str = json_str.replace('\n', ' ').replace('\r', '')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            # Try regex extraction for each expected field
            result = {}
            for field in ["label", "enriched_description", "yearly_description"]:
                match = re.search(rf'"{field}":\s*"(.*?)"', json_str, re.DOTALL)
                if match:
                    result[field] = match.group(1).strip()
            return result if result else None
        except:
            pass
    return None

## Load Topic Word Evolution Data

In [6]:
def load_topic_words(model: str, subject: str) -> pd.DataFrame:
    """Load topic_word_evolution.csv for a given model and subject."""
    path = BASE_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    if not path.exists():
        print(f"  [WARNING] File not found: {path}")
        return None
    df = pd.read_csv(path)
    print(f"  Loaded {len(df)} rows from {path}")
    return df

# Quick check
for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        path = BASE_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
        exists = "✓" if path.exists() else "✗"
        print(f"  {exists} {model}/{subject}")

  ✓ lda/cs
  ✓ lda/math
  ✓ lda/physics
  ✓ dtm/cs
  ✓ dtm/math
  ✓ dtm/physics
  ✓ bertopic/cs
  ✓ bertopic/math
  ✓ bertopic/physics
  ✓ top2vec/cs
  ✓ top2vec/math
  ✓ top2vec/physics
  ✓ topicGpt/cs
  ✓ topicGpt/math
  ✓ topicGpt/physics


---
## Step 1: Overall Label & Enriched Description

For each topic, combine **all top words across all years** into a single set, then ask the LLM to produce:
- A concise **label** (2-5 words)
- An **enriched description** (3-5 sentences describing the topic's scope)

In [7]:
LABEL_SYSTEM_PROMPT = """You are an expert academic topic analyst specializing in scientific literature.
Given a set of representative keywords from a topic discovered across multiple years of academic papers,
provide a concise label and a rich description for this topic.

OUTPUT RULES:
1. Return ONLY valid JSON: {"label": "...", "enriched_description": "..."}
2. The "label" must be 2-5 words, concise and descriptive.
3. The "enriched_description" must be 3-5 sentences describing the topic's scope, key methods, and applications in academic research.
4. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
5. If you use quotes inside values, use 'single quotes' so the JSON doesn't break.
6. Keep the entire description on ONE SINGLE LINE. No newlines inside the JSON value."""

LABEL_USER_TEMPLATE = """Topic ID: {topic_id}
Subject Area: {subject}

Below are all the representative keywords for this topic, collected across multiple years of academic papers:

{all_words}

Based on these keywords, provide a concise label and a rich academic description for this topic.
Return ONLY valid JSON: {{"label": "...", "enriched_description": "..."}}"""

In [8]:
def get_overall_labels(df: pd.DataFrame, model: str, subject: str) -> pd.DataFrame:
    """Step 1: Generate overall label + enriched description for each topic."""
    checkpoint = load_checkpoint("overall_labels", model, subject)
    if checkpoint is not None:
        print(f"  Loaded {len(checkpoint)} labels from checkpoint")
        return pd.DataFrame(checkpoint)
        
    if model == "topicGpt":
        print(f"  [topicGpt] Loading existing labels and enriched descriptions from enrichment.pkl...")
        from pathlib import Path
        enrich_path = Path(f"../../models/topicGpt/{subject}/enrichment.pkl")
        assign_path = Path(f"../../results/topicGpt/modeling/{subject}/topicgpt_assignments.csv")
        
        mapping = {}
        if assign_path.exists():
            try:
                mapping_df = pd.read_csv(assign_path)
                mapping = dict(zip(mapping_df["topic_id"], mapping_df["original_topic_id"]))
            except Exception as e:
                print(f"  [Warning] Failed to load original_topic_id mapping: {e}")
                
        if enrich_path.exists():
            import pickle
            with open(enrich_path, "rb") as f:
                enrich_data = pickle.load(f).get("enriched_topics", {})
                
            results = []
            topic_ids = sorted(df["topic_id"].unique())
            for topic_id in topic_ids:
                original_id = mapping.get(topic_id, topic_id)
                if original_id in enrich_data:
                    info = enrich_data[original_id]
                    results.append({
                        "topic_id": topic_id,
                        "label": info.get("label", f"Topic_{topic_id}"),
                        "enriched_description": info.get("enriched_description", info.get("description", "No description available."))
                    })
                else:
                    results.append({
                        "topic_id": topic_id,
                        "label": f"Topic_{topic_id}",
                        "enriched_description": "No description available."
                    })
            
            save_checkpoint(results, "overall_labels", model, subject)
            return pd.DataFrame(results)
        else:
            print(f"  [Warning] enrichment.pkl not found at {enrich_path}, falling back to LLM.")
    
    # Group by topic_id, collect all words across years
    topic_groups = df.groupby("topic_id")
    topic_ids = sorted(df["topic_id"].unique())
    
    results = []
    
    for topic_id in tqdm(topic_ids, desc=f"Labeling {model}/{subject}"):
        group = topic_groups.get_group(topic_id)
        
        # Collect all words across all years, deduplicate while preserving order
        all_words = []
        seen = set()
        for _, row in group.iterrows():
            words = [w.strip() for w in str(row["top_words"]).split(",")]
            for w in words:
                if w and w not in seen:
                    all_words.append(w)
                    seen.add(w)
        
        words_str = ", ".join(all_words)
        
        user_prompt = LABEL_USER_TEMPLATE.format(
            topic_id=topic_id,
            subject=subject,
            all_words=words_str
        )
        
        response = call_llm(LABEL_SYSTEM_PROMPT, user_prompt)
        parsed = clean_and_parse_json(response)
        
        label = f"Topic_{topic_id}"
        enriched_desc = "No description available."
        
        if parsed:
            label = parsed.get("label", label)
            enriched_desc = parsed.get("enriched_description", enriched_desc)
        else:
            print(f"  [Warning] Parse failed for topic {topic_id}")
        
        results.append({
            "topic_id": topic_id,
            "label": label,
            "enriched_description": enriched_desc
        })
        
        # Checkpoint every 20 topics
        if len(results) % 20 == 0:
            save_checkpoint(results, "overall_labels", model, subject)
    
    # Final save
    save_checkpoint(results, "overall_labels", model, subject)
    return pd.DataFrame(results)

In [9]:
# Run Step 1 for all models and subjects
all_labels = {}

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        print(f"\n{'='*60}")
        print(f"STEP 1 — LABELING: {model.upper()} / {subject.upper()}")
        print(f"{'='*60}")
        
        df = load_topic_words(model, subject)
        if df is None:
            continue
        
        labels_df = get_overall_labels(df, model, subject)
        all_labels[(model, subject)] = labels_df
        
        # Save to CSV
        out_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        labels_df.to_csv(out_path, index=False)
        print(f"  Saved {len(labels_df)} labels to {out_path}")
        
        # Preview
        print(f"\n  Preview (first 5):")
        for _, row in labels_df.head().iterrows():
            print(f"    [{row['topic_id']}] {row['label']}: {row['enriched_description'][:100]}...")


STEP 1 — LABELING: LDA / CS
  Loaded 1811 rows from ../../results/lda/temporal/cs/topic_word_evolution.csv


Labeling lda/cs:   0%|          | 0/75 [00:00<?, ?it/s]

Labeling lda/cs:  23%|██▎       | 17/75 [00:40<02:31,  2.61s/it]

  [Warning] Parse failed for topic 16


Labeling lda/cs:  27%|██▋       | 20/75 [00:47<02:05,  2.28s/it]

...

  [Warning] Parse failed for topic 206


Labeling top2vec/physics: 100%|█████████▉| 209/210 [08:00<00:02,  2.48s/it]

  [Warning] Parse failed for topic 208


Labeling top2vec/physics: 100%|██████████| 210/210 [08:03<00:00,  2.30s/it]

  [Warning] Parse failed for topic 209
  Checkpoint saved: ../../models/labeling/top2vec/physics/overall_labels.pkl
  Saved 210 labels to ../../results/top2vec/temporal/physics/topic_labels.csv

  Preview (first 5):
    [0] Density Functional Theory Variational Methods: This topic focuses on the application of density functional theory (DFT) and variational quantum met...
    [1] Multiscale network modeling in hydrological and biological systems: This topic examines the structural and functional scaling behaviors of complex networks across river...
    [2] Nonlinear Optical Soliton Dynamics in Fiber Lasers: This topic focuses on the study of solitons—self-reinforcing, stable light pulses—generated through ...
    [3] Quantum Entanglement-Based Networks: This field explores the fundamental and applied aspects of quantum entanglement, particularly in pho...
    [4] Multiscale Molecular Modeling & Machine Learning: This interdisciplinary field integrates high-precision molecular simulatio

---
## Step 2: Per-Year Simple Description

For each topic and each year, take the top words for **that specific year** and generate
a simple 1-2 sentence description of what the topic looks like in that year.

In [10]:
YEARLY_SYSTEM_PROMPT = """You are an expert academic topic analyst.
Given a topic label and the representative keywords from a specific year,
write a simple 1-2 sentence description of what this topic focused on in that year.

OUTPUT RULES:
1. Return ONLY valid JSON: {"yearly_description": "..."}
2. The description should be 1-2 sentences, plain and concise.
3. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
4. If you use quotes inside values, use 'single quotes'.
5. Keep the description on ONE SINGLE LINE."""

YEARLY_USER_TEMPLATE = """Topic Label: {label}
Subject Area: {subject}
Year: {year}

Keywords for this topic in {year}:
{words}

Write a simple 1-2 sentence description of what this topic focused on in {year}.
Return ONLY valid JSON: {{"yearly_description": "..."}}"""

In [11]:
def get_yearly_descriptions(df: pd.DataFrame, labels_df: pd.DataFrame, model: str, subject: str) -> pd.DataFrame:
    """Step 2: Generate per-year simple descriptions for each topic."""
    checkpoint = load_checkpoint("yearly_descriptions", model, subject)
    if checkpoint is not None:
        print(f"  Loaded {len(checkpoint)} yearly descriptions from checkpoint")
        return pd.DataFrame(checkpoint)
    
    # Build label lookup
    label_map = dict(zip(labels_df["topic_id"], labels_df["label"]))
    
    results = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Yearly desc {model}/{subject}"):
        topic_id = row["topic_id"]
        year = row["year"]
        words = str(row["top_words"]).strip()
        label = label_map.get(topic_id, f"Topic_{topic_id}")
        
        user_prompt = YEARLY_USER_TEMPLATE.format(
            label=label,
            subject=subject,
            year=year,
            words=words
        )
        
        response = call_llm(YEARLY_SYSTEM_PROMPT, user_prompt)
        parsed = clean_and_parse_json(response)
        
        yearly_desc = "No description available."
        if parsed and "yearly_description" in parsed:
            yearly_desc = parsed["yearly_description"]
        else:
            print(f"  [Warning] Parse failed for topic {topic_id}, year {year}")
        
        results.append({
            "topic_id": topic_id,
            "year": year,
            "label": label,
            "yearly_description": yearly_desc
        })
        
        # Checkpoint every 50 rows
        if len(results) % 50 == 0:
            save_checkpoint(results, "yearly_descriptions", model, subject)
    
    # Final save
    save_checkpoint(results, "yearly_descriptions", model, subject)
    return pd.DataFrame(results)

In [12]:
# Run Step 2 for all models and subjects
all_yearly = {}

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        print(f"\n{'='*60}")
        print(f"STEP 2 — YEARLY DESCRIPTIONS: {model.upper()} / {subject.upper()}")
        print(f"{'='*60}")
        
        df = load_topic_words(model, subject)
        if df is None:
            continue
        
        # Load labels from Step 1 (either from all_labels or from saved CSV)
        if (model, subject) in all_labels:
            labels_df = all_labels[(model, subject)]
        else:
            label_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
            if label_path.exists():
                labels_df = pd.read_csv(label_path)
            else:
                print(f"  [ERROR] Labels not found. Run Step 1 first.")
                continue
        
        yearly_df = get_yearly_descriptions(df, labels_df, model, subject)
        all_yearly[(model, subject)] = yearly_df
        
        # Save to CSV
        out_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
        yearly_df.to_csv(out_path, index=False)
        print(f"  Saved {len(yearly_df)} yearly descriptions to {out_path}")
        
        # Preview
        print(f"\n  Preview (first 5):")
        for _, row in yearly_df.head().iterrows():
            print(f"    [{row['topic_id']}|{row['year']}] {row['label']}: {row['yearly_description'][:80]}...")


STEP 2 — YEARLY DESCRIPTIONS: LDA / CS
  Loaded 1811 rows from ../../results/lda/temporal/cs/topic_word_evolution.csv
  Checkpoint loaded: ../../models/labeling/lda/cs/yearly_descriptions.pkl
  Loaded 600 yearly descriptions from checkpoint
  Saved 600 yearly descriptions to ../../results/lda/temporal/cs/topic_yearly_descriptions.csv

  Preview (first 5):
    [0|2000] Multiscale Deep Image Processing: In 2000, the focus of multiscale deep image processing centered on advancing wav...
    [1|2000] Fintech-Retail Market Dynamics: In 2000, the topic of Fintech-Retail Market Dynamics primarily examined how emer...
    [2|2000] Collaborative Scholarly Ecosystems: In 2000, the focus was on designing and evaluating **collaborative tools** like ...
    [3|2000] Multilingual Text Processing Frameworks: In 2000, the field of multilingual text processing frameworks primarily explored...
    [4|2000] Advanced Web Information Retrieval Systems: In 2000, the focus on advanced web information retrie

Yearly desc lda/math:   0%|          | 0/1201 [00:00<?, ?it/s]

Yearly desc lda/math:   4%|▍         | 50/1201 [00:39<14:51,  1.29it/s]

  Checkpoint saved: ../../models/labeling/lda/math/yearly_descriptions.pkl


Yearly desc lda/math:   8%|▊         | 100/1201 [01:18<14:34,  1.26it/s]

...

  Checkpoint saved: ../../models/labeling/topicGpt/physics/yearly_descriptions.pkl


Yearly desc topicGpt/physics: 100%|█████████▉| 1750/1753 [21:34<00:02,  1.35it/s]

  Checkpoint saved: ../../models/labeling/topicGpt/physics/yearly_descriptions.pkl


Yearly desc topicGpt/physics: 100%|██████████| 1753/1753 [21:36<00:00,  1.35it/s]

  Checkpoint saved: ../../models/labeling/topicGpt/physics/yearly_descriptions.pkl
  Saved 1753 yearly descriptions to ../../results/topicGpt/temporal/physics/topic_yearly_descriptions.csv

  Preview (first 5):
    [0|2000] Plasma Beam Interactions: In 2000, research on plasma beam interactions primarily explored how high-energy...
    [1|2000] Uncertainty Propagation: In 2000, *Uncertainty Propagation* primarily explored how probabilistic methods—...
    [2|2000] Quantum Corrections: In 2000, the focus on quantum corrections to 'alpha' (fine-structure constant) a...
    [3|2000] DNA Conformational Dynamics: In 2000, the study of DNA conformational dynamics primarily explored how short l...
    [4|2000] Optical Trapping Cooling: In 2000, the field of optical trapping cooling primarily explored methods to coo...


---
## Summary

Print a summary of all generated files.

In [13]:
print("\n" + "="*60)
print("LABELING & ENRICHMENT COMPLETE")
print("="*60)

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        label_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        yearly_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
        
        l_status = f"✓ {pd.read_csv(label_path).shape[0]} topics" if label_path.exists() else "✗ missing"
        y_status = f"✓ {pd.read_csv(yearly_path).shape[0]} rows" if yearly_path.exists() else "✗ missing"
        
        print(f"  {model}/{subject}: labels={l_status}, yearly={y_status}")


LABELING & ENRICHMENT COMPLETE
  lda/cs: labels=✓ 75 topics, yearly=✓ 600 rows
  lda/math: labels=✓ 50 topics, yearly=✓ 1201 rows
  lda/physics: labels=✓ 50 topics, yearly=✓ 1274 rows
  dtm/cs: labels=✓ 50 topics, yearly=✓ 1300 rows
  dtm/math: labels=✓ 50 topics, yearly=✓ 1300 rows
  dtm/physics: labels=✓ 60 topics, yearly=✓ 1560 rows
  bertopic/cs: labels=✓ 261 topics, yearly=✓ 4328 rows
  bertopic/math: labels=✓ 150 topics, yearly=✓ 3572 rows
  bertopic/physics: labels=✓ 232 topics, yearly=✓ 5162 rows
  top2vec/cs: labels=✓ 259 topics, yearly=✓ 5195 rows
  top2vec/math: labels=✓ 209 topics, yearly=✓ 5134 rows
  top2vec/physics: labels=✓ 210 topics, yearly=✓ 5136 rows
  topicGpt/cs: labels=✓ 138 topics, yearly=✓ 2221 rows
  topicGpt/math: labels=✓ 63 topics, yearly=✓ 1551 rows
  topicGpt/physics: labels=✓ 74 topics, yearly=✓ 1753 rows
